# Bayesian model comparison

Frequentist model selection leans on cross-validation and information criteria. The Bayesian view
compares models by how well they predicted the data, integrated over parameter uncertainty. This
notebook computes a Bayes factor in closed form for a coin-fairness question, reads it on the Jeffreys
scale, then turns to the widely applicable information criterion (WAIC) and leave-one-out
cross-validation for models where the marginal likelihood is not analytic, using a pooled-versus-grouped
mean as the running example.


## The marginal likelihood is an automatic Occam's razor

The Bayes factor between models is the ratio of their marginal likelihoods,
$\mathrm{BF}_{10}=p(x\mid M_1)/p(x\mid M_0)$ with $p(x\mid M)=\int p(x\mid\theta,M)\pi(\theta\mid M)d\theta$.

Theorem (Bayesian Occam's razor). The marginal likelihood penalizes complexity automatically. A model
with more parameters spreads its prior over a larger space, so it must assign lower prior density to the
region that fits the data; unless the extra flexibility is needed, the simpler model achieves a higher
marginal likelihood. The Laplace approximation makes this explicit:
$\log p(x\mid M)\approx \ell(\hat\theta)-\tfrac{k}{2}\log n$, the BIC penalty.

Theorem (WAIC and LOO agree asymptotically; Watanabe). The widely applicable information criterion equals
leave-one-out cross-validation up to $o(1)$, and both estimate the expected log predictive density on new
data. So the Bayesian (WAIC) and cross-validatory (LOO) routes to model assessment coincide in the limit,
which the applied section confirms numerically.

Counterexample (Lindley's paradox and prior sensitivity). The Bayes factor depends on the prior even
asymptotically. With a very diffuse prior on the alternative, a result that a frequentist test finds
significant can yield a Bayes factor favoring the null, because the diffuse prior wasted its mass.
Marginal-likelihood model choice is therefore sensitive to the prior in a way that parameter estimation
(which is not, by Bernstein-von Mises) is not; the prior on the extra parameters must be chosen with care.

## 1. Bayes factor from the marginal likelihood

The marginal likelihood is the probability of the data under a model after integrating out its
parameters. For a coin, compare a point-null model (the coin is fair, p = 0.5) against an alternative
with a uniform prior on p. Both marginal likelihoods are closed form, and their ratio is the Bayes
factor.


In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import betaln
rng = np.random.RandomState(0)
n, k = 100, 64                                          # 64 heads in 100 flips
log_m0 = k * np.log(0.5) + (n - k) * np.log(0.5)        # fair-coin model
log_m1 = betaln(k + 1, n - k + 1) - betaln(1, 1)        # uniform prior on p
bf10 = np.exp(log_m1 - log_m0)
print('data: %d heads in %d flips' % (k, n))
print('Bayes factor BF10 (biased vs fair) = %.1f' % bf10)

data: 64 heads in 100 flips
Bayes factor BF10 (biased vs fair) = 6.3


## 2. The Jeffreys scale

A Bayes factor is read on a standard interpretive scale: values past 10 are strong evidence, past 100
decisive. We classify the result and confirm it tracks intuition as the data grow more lopsided.


In [2]:
def jeffreys(bf):
    for t, label in [(100, 'decisive'), (30, 'very strong'), (10, 'strong'),
                     (3, 'substantial'), (1, 'weak')]:
        if bf >= t:
            return label
    return 'supports the null'
for kk in [50, 55, 60, 64, 70]:
    lm0 = n * np.log(0.5); lm1 = betaln(kk + 1, n - kk + 1) - betaln(1, 1)
    bf = np.exp(lm1 - lm0)
    print('%d/%d heads -> BF10 = %7.1f  (%s)' % (kk, n, bf, jeffreys(bf)))

50/100 heads -> BF10 =     0.1  (supports the null)
55/100 heads -> BF10 =     0.2  (supports the null)
60/100 heads -> BF10 =     0.9  (supports the null)
64/100 heads -> BF10 =     6.3  (substantial)
70/100 heads -> BF10 =   427.3  (decisive)


## 3. WAIC when the marginal likelihood is intractable

For most models the marginal likelihood has no closed form. WAIC estimates out-of-sample predictive
accuracy from the posterior: the log pointwise predictive density minus an effective-parameter penalty
computed from the variance of the pointwise log-likelihood across posterior draws. We compare a model
that pools two groups into one mean against a model that gives each group its own mean.


In [3]:
g = np.r_[np.zeros(60), np.ones(60)].astype(int)
y = np.r_[rng.normal(0.0, 1, 60), rng.normal(1.2, 1, 60)]   # the groups truly differ
sigma = 1.0
def posterior_mu(yy, draws=4000):
    post_var = 1 / (1 / 100 + len(yy) / sigma ** 2)         # weak N(0,10^2) prior, known sigma
    post_mean = post_var * (yy.sum() / sigma ** 2)
    return rng.normal(post_mean, np.sqrt(post_var), draws)
def waic(loglik):                                           # loglik: (draws, n_obs)
    lppd = np.log(np.exp(loglik).mean(0)).sum()
    p_w = loglik.var(0).sum()
    return -2 * (lppd - p_w), p_w
from scipy.stats import norm
mu_pool = posterior_mu(y)
ll_pool = norm.logpdf(y[None, :], mu_pool[:, None], sigma)
mu0, mu1 = posterior_mu(y[g == 0]), posterior_mu(y[g == 1])
ll_grp = np.where(g[None, :] == 0, norm.logpdf(y[None, :], mu0[:, None], sigma),
                  norm.logpdf(y[None, :], mu1[:, None], sigma))
w_pool, p_pool = waic(ll_pool); w_grp, p_grp = waic(ll_grp)
print('pooled  model: WAIC = %.1f  (effective params %.1f)' % (w_pool, p_pool))
print('grouped model: WAIC = %.1f  (effective params %.1f)' % (w_grp, p_grp))
print('lower WAIC wins: the grouped model predicts better because the groups really differ.')

pooled  model: WAIC = 405.3  (effective params 1.5)
grouped model: WAIC = 355.4  (effective params 2.2)
lower WAIC wins: the grouped model predicts better because the groups really differ.


## 4. Leave-one-out cross-validation

WAIC approximates leave-one-out cross-validation. We can also compute LOO directly here by importance
weighting each posterior draw, and confirm the two agree and pick the same model.


In [4]:
def loo(loglik):
    # importance-sampling LOO: weight by 1/likelihood of the held-out point
    w = np.exp(-loglik); w /= w.sum(0, keepdims=True)
    elpd = np.log((w * np.exp(loglik)).sum(0)).sum()
    return -2 * elpd
print('pooled  model: LOO deviance = %.1f' % loo(ll_pool))
print('grouped model: LOO deviance = %.1f' % loo(ll_grp))
print('LOO agrees with WAIC: both prefer the grouped model.')

pooled  model: LOO deviance = 405.3
grouped model: LOO deviance = 355.4
LOO agrees with WAIC: both prefer the grouped model.


## References

- Kass, R. & Raftery, A. (1995). Bayes factors. JASA.
- Jeffreys, H. (1961). Theory of Probability, 3rd ed. Oxford University Press.
- Watanabe, S. (2010). Asymptotic equivalence of Bayes cross validation and widely applicable information criterion. JMLR.
- Vehtari, A., Gelman, A. & Gabry, J. (2017). Practical Bayesian model evaluation using leave-one-out cross-validation and WAIC. Statistics and Computing. https://doi.org/10.1007/s11222-016-9696-4


## Exercises

1. Show the sensitivity of the Bayes factor to the prior on p (Lindley's paradox): widen the alternative prior and watch the Bayes factor shift, even though the data are unchanged.
2. Implement Pareto-smoothed importance sampling for LOO (PSIS-LOO) and use the Pareto k diagnostic to flag observations where the plain importance-sampling LOO above is unreliable.
3. Add a third model (a hierarchical partial-pooling mean) and rank all three by WAIC and LOO; relate the ranking to the shrinkage in the hierarchical fit.
4. Compare the WAIC and LOO rankings to AIC, BIC, and k-fold cross-validation on the same models and discuss when the Bayesian and frequentist criteria disagree.
